# DAY 7: Production-Style API Deployment & Smoke Testing

## 🎯 Learning Objectives
In this final milestone, you package your fine-tuned and merged LLM into a **usable, production-oriented API service**.

By the end of this day, you will be able to execute and explain:
1. **Production LLM Serving Architecture**: FastAPI + Uvicorn lifecycle management with Lifespan context handlers.
2. **Preloading Weights**: Why model weights must **never** be loaded inside request endpoints.
3. **Schema Validation**: Using Pydantic models for strict request parsing, response formatting, and error handling.
4. **Containerization with Docker**: Writing multi-stage Dockerfiles with healthchecks, non-root users, and environment flags.
5. **Deployment Smoke Testing**: End-to-end verification (`docker build` $\to$ `/health` $\to$ `/predict` $\to$ valid JSON response).
6. **Cloud GPU Deployment**: Deploying across RunPod, Modal, AWS EC2, or Hugging Face Spaces.

## 📐 The Production Serving Architecture

```
Client Request (JSON HTTP POST)
         ↓
   FastAPI Gateway (api/main.py)
         ↓  [Pydantic Request Validation: api/schemas.py]
   Inference Engine (api/inference.py)
         ↓  [Loaded once during server startup lifespan]
   Merged Standalone Fine-Tuned LLM (PyTorch / CUDA)
         ↓  [Autoregressive Token Generation]
   Output Parser & Schema Validator (src/data/validator.py)
         ↓  [Pydantic Response Serialization]
Client Receives Validated JSON with Latency Metrics
```

In [1]:
!pip install -q fastapi uvicorn pydantic httpx pytest

### Step 1: Test API with FastAPI TestClient (Local Smoke Test)

In [3]:
import os
import sys

# 1. Install FastAPI & TestClient dependencies
!pip install -q fastapi uvicorn httpx pydantic

# 2. Update to latest repository code
if not os.path.exists("/content/Fine-Tuning-PoCs"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
else:
    %cd /content/Fine-Tuning-PoCs
    !git pull origin main

# 3. Change directory into project root
%cd /content/Fine-Tuning-PoCs/llm-finetuning-production

# 4. Add project root to sys.path so Python finds 'api' and 'src'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("✅ Current Directory:", os.getcwd())
print("✅ Found 'api' package:", os.path.exists("api/main.py"))
print("✅ Found 'models/merged':", os.path.exists("models/merged/qwen-1.5b-order-extractor"))


/content/Fine-Tuning-PoCs
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 14 (delta 10), reused 14 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 6.50 KiB | 1.08 MiB/s, done.
From https://github.com/sagarsambhwani/Fine-Tuning-PoCs
 * branch            main       -> FETCH_HEAD
   80313bb..e1dbf7e  main       -> origin/main
Updating 80313bb..e1dbf7e
Fast-forward
 llm-finetuning-production/README.md                |  55 ++++--
 .../notebooks/day06_quantization.ipynb             | 187 ++++++++++++++++++++-
 .../reports/day06_quantization_report.md           |  88 ++++++++++
 3 files changed, 308 insertions(+), 22 deletions(-)
 create mode 100644 llm-finetuning-production/reports/day06_quantization_report.md
/content/Fine-Tuning-PoCs/llm-finetuning-production
✅ Current Directory: /content/Fine-Tuning-PoCs/llm-finetuning-production
✅ Found 'api' package: True
✅ Found 'models

In [4]:
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

# 1. Test Health Endpoint
health_res = client.get("/health")
print("Health Endpoint Response:", health_res.status_code, health_res.json())
assert health_res.status_code == 200

# 2. Test Predict Endpoint with Sample Natural Language Order
payload = {
    "text": "John ordered 3 laptops for $2400 and wants delivery on Friday.",
    "temperature": 0.0
}

print("\nSending test POST /predict payload...")
predict_res = client.post("/predict", json=payload)
print("Predict Endpoint Status:", predict_res.status_code)
print("Response JSON:")
print(predict_res.json())

2026-08-26 11:03:23 | INFO     | api_inference_manager | Loading model into inference engine: models/merged/qwen-1.5b-order-extractor (adapter: None, device: None)


INFO:api_inference_manager:Loading model into inference engine: models/merged/qwen-1.5b-order-extractor (adapter: None, device: None)


2026-08-26 11:03:23 | INFO     | inference_engine | Initializing Inference Engine on device=cuda, dtype=torch.float16


INFO:inference_engine:Initializing Inference Engine on device=cuda, dtype=torch.float16


2026-08-26 11:03:23 | INFO     | inference_engine | Loading tokenizer from models/merged/qwen-1.5b-order-extractor...


...

INFO:inference_engine:Loading weights from models/merged/qwen-1.5b-order-extractor...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-08-26 11:03:35 | INFO     | inference_engine | Model loaded successfully into memory.


INFO:inference_engine:Model loaded successfully into memory.


Health Endpoint Response: 200 {'status': 'healthy', 'model': 'models/merged/qwen-1.5b-order-extractor', 'device': 'cuda'}

Sending test POST /predict payload...
Predict Endpoint Status: 200
Response JSON:
{'data': {'customer': 'John', 'quantity': 3, 'product': 'laptops', 'amount': 2400.0, 'delivery_day': 'Friday'}, 'raw_response': '{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}', 'is_valid_json': True, 'error': None, 'latency_ms': 2556.69, 'model_name': 'models/merged/qwen-1.5b-order-extractor'}


### Step 2: Verify Pydantic Schema Validation & Error Handling

In [5]:
# Test invalid short string (triggering Pydantic min_length=3 validation error 422)
invalid_res = client.post("/predict", json={"text": "ab"})
print("Expected 422 for invalid short text:", invalid_res.status_code)
assert invalid_res.status_code == 422

Expected 422 for invalid short text: 422


### Step 3: Production Docker & Smoke Test Instructions

In [6]:
smoke_test_script = '''
# 1. Build Docker Container
docker build -t llm-json-extractor:latest .

# 2. Run Container with GPU
docker run --gpus all -d -p 8000:8000 --name json-api llm-json-extractor:latest

# 3. Verify Healthcheck
curl -s http://localhost:8000/health

# 4. Verify Live Inference
curl -s -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Sarah bought 5 monitors for $1500 and requested delivery on Monday."}'

# 5. Cleanup
docker stop json-api && docker rm json-api
'''
print("Docker Smoke Test Commands:")
print(smoke_test_script)

Docker Smoke Test Commands:

# 1. Build Docker Container
docker build -t llm-json-extractor:latest .

# 2. Run Container with GPU
docker run --gpus all -d -p 8000:8000 --name json-api llm-json-extractor:latest

# 3. Verify Healthcheck
curl -s http://localhost:8000/health

# 4. Verify Live Inference
curl -s -X POST http://localhost:8000/predict   -H "Content-Type: application/json"   -d '{"text": "Sarah bought 5 monitors for $1500 and requested delivery on Monday."}'

# 5. Cleanup
docker stop json-api && docker rm json-api



---
## 🎓 Final Resume Readiness Evaluation & Interview Mastery

### 🏆 Resume Skills You Can Honestly Claim (After Running the Pipeline):
`LLM Fine-Tuning · LoRA · QLoRA · PEFT · SFT · Hugging Face Transformers · TRL · bitsandbytes · Quantization · LLM Evaluation · Model Serving · FastAPI · Docker`

### 💼 10 Essential Interview Questions to Practice:
1. **What is the difference between LoRA and QLoRA?**
   - *Answer*: LoRA trains low-rank adapter matrices $BA$ alongside frozen 16-bit base weights. QLoRA quantizes the base model weights to 4-bit NormalFloat (NF4) with Double Quantization and uses Paged 8-bit optimizers, reducing memory by ~70% while keeping adapter training in 16-bit.
2. **Why do we freeze base weights $W_0$ instead of fine-tuning everything?**
   - *Answer*: Freezing base weights drastically reduces optimizer states (which take 8 bytes/param in AdamW), eliminates catastrophic forgetting, and makes adapter storage lightweight (~20MB vs ~3GB).
3. **What is the exact LoRA mathematical update equation?**
   - *Answer*: $W' = W_0 + \left(\frac{\alpha}{r}\right) B A$, where $A \sim \mathcal{N}(0, \sigma^2)$ and $B = 0$ at initialization.
4. **What is NormalFloat4 (NF4) and why is it better than INT4?**
   - *Answer*: NF4 establishes quantile bins matching the normal distribution of neural network weights, ensuring equal information density per bin and lower reconstruction error than uniform INT4.
5. **Why do we set prompt tokens to `-100` in SFT?**
   - *Answer*: In PyTorch CrossEntropyLoss, `-100` is the default `ignore_index`. Masking prompt tokens ensures the model only calculates loss and gradients on the target completion.
6. **How did you prevent data leakage in your fine-tuning pipeline?**
   - *Answer*: We separated train, validation, and test dataset generation with deterministic seeds and programmatically verified zero overlap across input prompt strings.
7. **How do you evaluate whether a fine-tuned model actually improved over baseline?**
   - *Answer*: By evaluating both models on the exact same held-out test split using structured metrics (JSON validity rate, exact match rate, and field-level accuracy) under deterministic greedy decoding ($T=0.0$).
8. **What does `merge_and_unload()` do?**
   - *Answer*: It algebraically fuses the adapter matrices into the base model weights, creating a standalone model without PEFT library dependencies or runtime latency overhead.
9. **Why must model loading happen in the FastAPI lifespan handler rather than the request route?**
   - *Answer*: Loading model weights into VRAM takes seconds and several gigabytes of memory. Lifespan handlers load weights once upon server startup, allowing request handlers to perform fast, non-blocking inference.
10. **How would you scale this deployment in a cloud environment?**
    - *Answer*: Containerize with Docker, deploy across auto-scaling GPU instances (e.g. AWS ECS/EKS with g4dn or RunPod/Modal), put behind an NGINX reverse proxy or load balancer, and integrate continuous latency/VRAM monitoring via Prometheus and Grafana.

### 🗺️ Next 30-Day Learning Plan
- **Days 1–10**: Direct Preference Optimization (DPO) and RLHF for steering model tone and refusal handling.
- **Days 11–20**: High-throughput inference engines: vLLM with PagedAttention and TensorRT-LLM.
- **Days 21–30**: Agentic tool calling and function fine-tuning (teaching LLMs to emit strict API calls with schema constraints).